# Avaliação no split de teste

Avalia modelos no split de teste, reportando precisão, recall, mAP@50 e mAP@50-95. Compara o baseline do TCC1 (`runs/baseline/best.pt`) contra o modelo treinado com o pipeline novo, pra validar se o pré-processamento realmente melhora o resultado.

In [ ]:
from pathlib import Path
import yaml
import tempfile

from ultralytics import YOLO

DATASET_YAML = Path("../../../../data/dataset.yaml")
PROJECT = Path("../runs")
BASELINE_MODEL = PROJECT / "baseline" / "best.pt"

# Mesmo esquema de resolução de path do 02_train.ipynb -- o dataset.yaml
# versionado tem "path: ." como placeholder, sobrescrito aqui em runtime.
with open(DATASET_YAML) as f:
    dataset_cfg = yaml.safe_load(f)
dataset_cfg["path"] = str(DATASET_YAML.parent.resolve())

RESOLVED_DATASET_YAML = Path(tempfile.gettempdir()) / "cytoml_dataset_resolved.yaml"
with open(RESOLVED_DATASET_YAML, "w") as f:
    yaml.safe_dump(dataset_cfg, f)

print(f"Dataset: {RESOLVED_DATASET_YAML} (path={dataset_cfg['path']})")


Função auxiliar: carrega um modelo e roda a validação no split `test` do `dataset.yaml`, retornando as métricas principais.

In [ ]:
def evaluate_model(model_path: Path, label: str) -> dict:
    if not model_path.exists():
        print(f"[{label}] modelo não encontrado em {model_path}, pulando.")
        return None

    model = YOLO(model_path)
    metrics = model.val(data=RESOLVED_DATASET_YAML, split="test")

    result = {
        "label": label,
        "precision": metrics.box.mp,
        "recall": metrics.box.mr,
        "map50": metrics.box.map50,
        "map50_95": metrics.box.map,
    }
    print(
        f"[{label}] precision={result['precision']:.4f} recall={result['recall']:.4f} "
        f"mAP50={result['map50']:.4f} mAP50-95={result['map50_95']:.4f}"
    )
    return result


Avalia o baseline (peso original do TCC1) no split de teste.

In [ ]:
baseline_result = evaluate_model(BASELINE_MODEL, "Baseline (TCC1)")

Avalia o modelo mais recente treinado com o pipeline novo (se já existir -- rode `02_train.ipynb` primeiro). Procura a run mais recente em `../runs/`, ignorando a pasta `baseline/`.

In [ ]:
candidate_runs = [
    p for p in PROJECT.iterdir()
    if p.is_dir() and p.name != "baseline" and (p / "weights" / "best.pt").exists()
]
latest_run = max(candidate_runs, key=lambda p: p.stat().st_mtime) if candidate_runs else None

if latest_run is None:
    print("Nenhum modelo treinado encontrado em ../runs/ ainda -- rode 02_train.ipynb primeiro.")
    new_result = None
else:
    print(f"Avaliando run mais recente: {latest_run.name}")
    new_result = evaluate_model(latest_run / "weights" / "best.pt", f"Novo ({latest_run.name})")


Compara os dois lado a lado.

In [ ]:
results = [r for r in (baseline_result, new_result) if r is not None]

if len(results) < 2:
    print("Preciso dos dois resultados (baseline + novo) pra comparar -- rode as células acima primeiro.")
else:
    print(f"{'':25s} {'precision':>10s} {'recall':>10s} {'mAP50':>10s} {'mAP50-95':>10s}")
    for r in results:
        print(f"{r['label']:25s} {r['precision']:>10.4f} {r['recall']:>10.4f} {r['map50']:>10.4f} {r['map50_95']:>10.4f}")

    delta = results[1]["map50_95"] - results[0]["map50_95"]
    print(f"\nDelta mAP50-95 (novo - baseline): {delta:+.4f}")
